# Projeto ETL de Futebol

## Importações

In [111]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

## Carregando a API KEY

In [112]:
load_dotenv()
api_key = os.getenv("API_KEY")

## Requisição

In [113]:
url = "https://v3.football.api-sports.io/leagues"

headers = {
    "x-apisports-key": api_key
}

resposta = requests.get(url, headers=headers)
print(resposta.status_code)

dados = resposta.json()
dados.keys()

200


dict_keys(['get', 'parameters', 'errors', 'results', 'paging', 'response'])

## Navegando pela a API

In [114]:
dados["response"]

for liga in dados["response"]:
    if "Brazil" in liga["country"]["name"]:
        print(liga["league"]["id"], liga["league"]["name"])

71 Serie A
72 Serie B
75 Serie C
76 Serie D
520 Acreano
73 Copa Do Brasil
74 Brasileiro Women
521 Amapaense
522 Amazonense
77 Alagoano
613 Baiano - 2
620 Cearense - 2
625 Carioca - 2
843 Copa Verde
475 Paulista - A1
476 Paulista - A2
477 Gaúcho - 1
478 Gaúcho - 2
740 Brasileiro U20 A
741 Brasileiro de Aspirantes
742 Copa Paulista
602 Baiano - 1
603 Paraibano
604 Catarinense - 1
605 Paulista - A3
606 Paranaense - 1
607 Roraimense
608 Maranhense
609 Cearense - 1
610 Brasiliense
611 Capixaba
612 Copa do Nordeste
614 Paranaense - 2
615 Rondoniense
616 Potiguar
617 Copa do Brasil U20
618 São Paulo Youth Cup
619 Mineiro - 2
621 Piauiense
622 Pernambucano - 1
623 Sul-Matogrossense
624 Carioca - 1
626 Sergipano
627 Paraense
628 Goiano - 1
629 Mineiro - 1
630 Matogrossense
631 Tocantinense
632 Supercopa do Brasil
851 Carioca A2
936 Catarinense - 2
1030 Goiano - 2
1035 Copa Rio
1036 Copa Santa Catarina
1037 Paraibano 2
1059 Recopa Catarinense
1062 Paulista - A4
1063 Copa Alagoas
1069 Goiano U20


## Pegando a Liga Brasileira Seria A 2024

In [115]:
url = "https://v3.football.api-sports.io/fixtures"

params = {
    "league": 71,
    "season": 2024
}

resposta = requests.get(
    url,
    headers=headers,
    params=params
)

print(resposta.status_code)
dados = resposta.json()
dados.keys()

dados["response"][0]

200


{'fixture': {'id': 1180355,
  'referee': 'Rodrigo José Pereira de Lima',
  'timezone': 'UTC',
  'date': '2024-04-13T21:30:00+00:00',
  'timestamp': 1713043800,
  'periods': {'first': 1713043800, 'second': 1713047400},
  'venue': {'id': 244,
   'name': 'Estádio José Pinheiro Borda',
   'city': 'Porto Alegre, Rio Grande do Sul'},
  'status': {'long': 'Match Finished',
   'short': 'FT',
   'elapsed': 90,
   'extra': None}},
 'league': {'id': 71,
  'name': 'Serie A',
  'country': 'Brazil',
  'logo': 'https://media.api-sports.io/football/leagues/71.png',
  'flag': 'https://media.api-sports.io/flags/br.svg',
  'season': 2024,
  'round': 'Regular Season - 1',
  'standings': True},
 'teams': {'home': {'id': 119,
   'name': 'Internacional',
   'logo': 'https://media.api-sports.io/football/teams/119.png',
   'winner': True},
  'away': {'id': 118,
   'name': 'Bahia',
   'logo': 'https://media.api-sports.io/football/teams/118.png',
   'winner': False}},
 'goals': {'home': 2, 'away': 1},
 'score': 

## Extraindo todas as partidas

In [116]:
partidas = []

for partida in dados["response"]:
    partida_extraida = {
        "data": partida["fixture"]["date"],
        "rodada": partida["league"]["round"],
        "mandante": partida["teams"]["home"]["name"],
        "visitante": partida["teams"]["away"]["name"],
        "gols_mandante": partida["goals"]["home"],
        "goas_visitantes": partida["goals"]["away"]
    }
    
    partidas.append(partida_extraida)

len(partidas)

partidas[:3]

[{'data': '2024-04-13T21:30:00+00:00',
  'rodada': 'Regular Season - 1',
  'mandante': 'Internacional',
  'visitante': 'Bahia',
  'gols_mandante': 2,
  'goas_visitantes': 1},
 {'data': '2024-04-13T21:30:00+00:00',
  'rodada': 'Regular Season - 1',
  'mandante': 'Criciuma',
  'visitante': 'Juventude',
  'gols_mandante': 1,
  'goas_visitantes': 1},
 {'data': '2024-04-14T00:00:00+00:00',
  'rodada': 'Regular Season - 1',
  'mandante': 'Fluminense',
  'visitante': 'RB Bragantino',
  'gols_mandante': 2,
  'goas_visitantes': 2}]

## Transformando em DataFrame (Pandas)

In [117]:
df = pd.DataFrame(partidas)
df.head()

,data,rodada,mandante,visitante,gols_mandante,goas_visitantes
0,2024-04-13T21:30:00+00:00,Regular Season - 1,Internacional,Bahia,2,1
1,2024-04-13T21:30:00+00:00,Regular Season - 1,Criciuma,Juventude,1,1
2,2024-04-14T00:00:00+00:00,Regular Season - 1,Fluminense,RB Bragantino,2,2
3,2024-04-14T00:00:00+00:00,Regular Season - 1,Sao Paulo,Fortaleza EC,1,2
4,2024-04-14T19:00:00+00:00,Regular Season - 1,Corinthians,Atletico-MG,0,0


## Tratando a Tabela

In [118]:
# Verificando valores duplicados
display(df.duplicated().sum())

#Tipos de dados
df.dtypes

# Transformando a coluna data em datetime
df["data"] = pd.to_datetime(df["data"])
df.dtypes

np.int64(0)

data               datetime64[us, UTC]
rodada                             str
mandante                           str
visitante                          str
gols_mandante                    int64
goas_visitantes                  int64
dtype: object

### Padronizando algumas informações

In [119]:
# Pegando os valores únicos
df["rodada"].unique()

# Extraindo o número da rodada
df["rodada"] = df["rodada"].str.extract(r"(\d+)$").astype(int)

df.rename(columns={"goas_visitantes": "gols_visitante"}, inplace=True)

df.info()

df

df["rodada"].value_counts().sort_index()

df[["gols_mandante", "gols_visitante"]].describe()

correcoes = {
    "Atletico Goianiense": "Atlético Goianiense",
    "Atletico Paranaense": "Athletico Paranaense",
    "Atletico-MG": "Atlético-MG",
    "Criciuma": "Criciúma",
    "Gremio": "Grêmio",
    "Sao Paulo": "São Paulo",
    "Vasco DA Gama": "Vasco da Gama",
    "Vitoria": "Vitória",
    "Cuiaba": "Cuiabá"
}

df["mandante"] = df["mandante"].replace(correcoes)
df["visitante"] = df["visitante"].replace(correcoes)

<class 'pandas.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype              
---  ------          --------------  -----              
 0   data            380 non-null    datetime64[us, UTC]
 1   rodada          380 non-null    int64              
 2   mandante        380 non-null    str                
 3   visitante       380 non-null    str                
 4   gols_mandante   380 non-null    int64              
 5   gols_visitante  380 non-null    int64              
dtypes: datetime64[us, UTC](1), int64(3), str(2)
memory usage: 25.5 KB


## Carregar os Dados no Banco de Dados

### Importação e estabelecendo a conexão

In [120]:
import sqlite3
conn = sqlite3.connect("brasileirao.db")

### Carregando os dados

In [121]:
df.to_sql(
    "partidas",
    conn,
    if_exists="replace",
    index=False
)

380

### Exibindo os dados dentro do DB

In [122]:
pd.read_sql("SELECT * FROM partidas LIMIT 5", conn)

,data,rodada,mandante,visitante,gols_mandante,gols_visitante
0,2024-04-13 21:30:00+00:00,1,Internacional,Bahia,2,1
1,2024-04-13 21:30:00+00:00,1,Criciúma,Juventude,1,1
2,2024-04-14 00:00:00+00:00,1,Fluminense,RB Bragantino,2,2
3,2024-04-14 00:00:00+00:00,1,São Paulo,Fortaleza EC,1,2
4,2024-04-14 19:00:00+00:00,1,Corinthians,Atlético-MG,0,0


### Corrigindo os nomes de Alguns Clubes

In [123]:
correcoes = {
    "Atletico Goianiense": "Atlético Goianiense",
    "Atletico Paranaense": "Athletico Paranaense",
    "Atletico-MG": "Atlético-MG",
    "Criciuma": "Criciúma",
    "Gremio": "Grêmio",
    "Sao Paulo": "São Paulo",
    "Vasco DA Gama": "Vasco da Gama",
    "Vitoria": "Vitória",
    "Cuiaba": "Cuiabá"
}

for antigo, novo in correcoes.items():
    conn.execute(
        "UPDATE partidas SET mandante = ? WHERE mandante = ?",
        (novo, antigo)
    )
    conn.execute(
        "UPDATE partidas SET visitante = ? WHERE visitante = ?",
        (novo, antigo)
    )

conn.commit()

### Total de gols na temporada

In [124]:
pd.read_sql("""
    SELECT 
        SUM(gols_mandante) + SUM(gols_visitante) AS total_gols
    FROM partidas;
""", conn
)

,total_gols
0,929


### Média de gols por partida

In [125]:
pd.read_sql("""
    SELECT 
        (SUM(gols_mandante) + SUM(gols_visitante)) / COUNT(*) AS media_gols
    FROM partidas;          
""", conn
)

,media_gols
0,2


### Ranking da artilharia por clube

In [126]:
pd.read_sql("""
    WITH gols_mandante AS (
        SELECT  
            mandante,
            SUM(gols_mandante) AS total_gols_mandante
        FROM partidas
        GROUP BY mandante
    ),
    
    gols_visitante AS (
        SELECT
            visitante,
            SUM(gols_visitante) AS total_gols_visitante
        FROM partidas
        GROUP BY visitante
    ),

    gols_clube AS (
        SELECT 
            m.mandante AS clube,
            m.total_gols_mandante + v.total_gols_visitante AS total_gols,
            ROW_NUMBER () OVER(ORDER BY m.total_gols_mandante + v.total_gols_visitante DESC) AS pos
        FROM gols_mandante m
        INNER JOIN gols_visitante v
            ON m.mandante = v.visitante
    )
    
    SELECT
        pos AS 'Posição',
        clube AS Clube,
        total_gols AS 'Total de Gols'
    FROM gols_clube
    ORDER BY pos;
""", conn
)

,Posição,Clube,Total de Gols
0,1,Flamengo,61
1,2,Palmeiras,60
2,3,Botafogo,59
3,4,Corinthians,54
4,5,Fortaleza EC,53
5,6,Internacional,53
6,7,São Paulo,53
7,8,Bahia,49
8,9,Juventude,48
9,10,Atlético-MG,47


### Classificação

In [127]:
pd.read_sql("""
    SELECT
        mandante AS Clube,
        
        SUM(CASE
            WHEN gols_mandante > gols_visitante THEN 3
            WHEN gols_mandante < gols_visitante THEN 0
            ELSE 1
        END) AS Pontos,
        
        SUM(CASE
            WHEN gols_mandante > gols_visitante THEN 1
            ELSE 0
        END) AS Vitorias,
        
        SUM(CASE
            WHEN gols_mandante < gols_visitante THEN 1
            ELSE 0
        END) AS Derrotas,
        
        SUM(CASE
            WHEN gols_mandante = gols_visitante THEN 1
            ELSE 0
        END) AS Empates,
        
        SUM(gols_mandante) AS Gols_Marcados,
        SUM(gols_visitante) AS Gols_Sofridos,
        SUM(gols_mandante) - SUM(gols_visitante) AS Saldo_de_Gol
    FROM partidas
    GROUP BY clube
    ORDER BY Pontos DESC;
""", conn
)

,Clube,Pontos,Vitorias,Derrotas,Empates,Gols_Marcados,Gols_Sofridos,Saldo_de_Gol
0,Fortaleza EC,47,14,0,5,32,8,24
1,Botafogo,41,12,2,5,31,13,18
2,São Paulo,39,12,4,3,31,18,13
3,Palmeiras,38,12,5,2,34,16,18
4,Internacional,38,11,3,5,30,15,15
5,Flamengo,38,11,3,5,30,21,9
6,Corinthians,38,10,1,8,37,18,19
7,Bahia,36,11,5,3,25,15,10
8,Vasco da Gama,35,10,4,5,29,24,5
9,Cruzeiro,35,10,4,5,29,16,13


In [128]:
pd.read_sql("""
    SELECT
        visitante AS Clube,
        SUM(CASE
            WHEN gols_visitante > gols_mandante THEN 3
            WHEN gols_visitante < gols_mandante THEN 0
            ELSE 1
        END) AS Pontos,
        SUM(CASE
            WHEN gols_visitante > gols_mandante THEN 1
            ELSE 0
        END) AS Vitorias,
        SUM(CASE
            WHEN gols_visitante < gols_mandante THEN 1
            ELSE 0
        END) AS Derrotas,
        SUM(CASE
            WHEN gols_visitante = gols_mandante THEN 1
            ELSE 0
        END) AS Empates,
        SUM(gols_visitante) AS Gols_Marcados,
        SUM(gols_mandante) AS Gols_Sofridos,
        SUM(gols_visitante) - SUM(gols_mandante) AS Saldo_de_Gol
    FROM partidas
    GROUP BY clube
    ORDER BY Pontos DESC;    
""", conn
)

,Clube,Pontos,Vitorias,Derrotas,Empates,Gols_Marcados,Gols_Sofridos,Saldo_de_Gol
0,Botafogo,38,11,3,5,28,16,12
1,Palmeiras,35,10,4,5,26,17,9
2,Flamengo,32,9,5,5,31,21,10
3,Internacional,27,7,6,6,23,21,2
4,Fortaleza EC,21,5,8,6,21,31,-10
5,Vitória,20,5,9,5,22,29,-7
6,São Paulo,20,5,9,5,22,25,-3
7,Atlético-MG,19,4,8,7,20,28,-8
8,Corinthians,18,5,11,3,17,27,-10
9,Fluminense,17,5,12,2,13,23,-10


In [129]:
pd.read_sql("""
    SELECT 
        Clube,
        SUM(Pontos) AS Pontos,
        SUM(Vitorias) AS Vitorias,
        SUM(Derrotas) AS Derrotas,
        SUM(Empates) AS Empates,
        SUM(Gols_Marcados) AS Gols_Marcados,
        SUM(Gols_Sofridos) AS Gols_Sofridos,
        SUM(Gols_Marcados) - SUM(Gols_Sofridos) AS Saldo_de_Gols
    FROM (
        SELECT
            mandante AS Clube,
            SUM(CASE
                WHEN gols_mandante > gols_visitante THEN 3
                WHEN gols_mandante < gols_visitante THEN 0
                ELSE 1
            END) AS Pontos,
            SUM(CASE
                WHEN gols_mandante > gols_visitante THEN 1
                ELSE 0
            END) AS Vitorias,
            SUM(CASE
                WHEN gols_mandante < gols_visitante THEN 1
                ELSE 0
            END) AS Derrotas,
            SUM(CASE
                WHEN gols_mandante = gols_visitante THEN 1
                ELSE 0
            END) AS Empates,
            SUM(gols_mandante) AS Gols_Marcados,
            SUM(gols_visitante) AS Gols_Sofridos,
            SUM(gols_mandante) - SUM(gols_visitante) AS Saldo_de_Gol
        FROM partidas
        GROUP BY Clube
        
        UNION ALL
            
        SELECT
            visitante AS Clube,
            SUM(CASE
                WHEN gols_visitante > gols_mandante THEN 3
                WHEN gols_visitante < gols_mandante THEN 0
                ELSE 1
            END) AS Pontos,
            SUM(CASE
                WHEN gols_visitante > gols_mandante THEN 1
                ELSE 0
            END) AS Vitorias,
            SUM(CASE
                WHEN gols_visitante < gols_mandante THEN 1
                ELSE 0
            END) AS Derrotas,
            SUM(CASE
                WHEN gols_visitante = gols_mandante THEN 1
                ELSE 0
            END) AS Empates,
            SUM(gols_visitante) AS Gols_Marcados,
            SUM(gols_mandante) AS Gols_Sofridos,
            SUM(gols_visitante) - SUM(gols_mandante) AS Saldo_de_Gol
        FROM partidas
        GROUP BY Clube
    )
    GROUP BY Clube
    ORDER BY Pontos DESC, Saldo_de_Gols DESC, Gols_Marcados DESC;
""", conn
)

,Clube,Pontos,Vitorias,Derrotas,Empates,Gols_Marcados,Gols_Sofridos,Saldo_de_Gols
0,Botafogo,79,23,5,10,59,29,30
1,Palmeiras,73,22,9,7,60,33,27
2,Flamengo,70,20,8,10,61,42,19
3,Fortaleza EC,68,19,8,11,53,39,14
4,Internacional,65,18,9,11,53,36,17
5,São Paulo,59,17,13,8,53,43,10
6,Corinthians,56,15,12,11,54,45,9
7,Bahia,53,15,15,8,49,49,0
8,Cruzeiro,52,14,14,10,43,41,2
9,Vasco da Gama,50,14,16,8,43,56,-13


### Resumo Geral do Campeonato

In [130]:
pd.read_sql("""
    SELECT
        COUNT(*) AS Total_Partidas,
        SUM(gols_mandante) + SUM(gols_visitante) AS Total_Gols,
        AVG(gols_mandante) + AVG(gols_visitante) AS Media_Gols,
        MAX(
            CASE
                WHEN gols_mandante > gols_visitante THEN gols_mandante - gols_visitante
                WHEN gols_visitante > gols_mandante THEN gols_visitante - gols_mandante
                ELSE 0
            END
        ) AS Maior_Goleada,
        MAX(gols_mandante + gols_visitante) AS Maior_Gols_Partida
    FROM partidas;
""", conn
)

,Total_Partidas,Total_Gols,Media_Gols,Maior_Goleada,Maior_Gols_Partida
0,380,929,2.444737,5,8


### Desempenho Mandante vs Visitante

In [131]:
pd.read_sql("""
WITH vitorias_mandante AS (
    SELECT
        mandante AS Clube,
        SUM(CASE
            WHEN gols_mandante > gols_visitante THEN 1
            ELSE 0
        END) AS Vitorias_em_Casa
    FROM partidas
    GROUP BY Clube
),

vitorias_visitante AS (
    SELECT
        visitante AS Clube,
        SUM(CASE
            WHEN gols_visitante > gols_mandante THEN 1
            ELSE 0
        END) AS Vitorias_fora
    FROM partidas
    GROUP BY Clube
)

    SELECT
        m.Clube,
        m.Vitorias_em_Casa,
        f.Vitorias_fora,
        m.Vitorias_em_Casa - f.Vitorias_fora AS Diferenca_Vitorias
    FROM vitorias_mandante m
    INNER JOIN vitorias_visitante f
        ON m.Clube = f.Clube
    ORDER BY m.Vitorias_em_Casa - f.Vitorias_fora DESC;
""", conn
)

,Clube,Vitorias_em_Casa,Vitorias_fora,Diferenca_Vitorias
0,Fortaleza EC,14,5,9
1,Bahia,11,4,7
2,São Paulo,12,5,7
3,Cruzeiro,10,4,6
4,Grêmio,9,3,6
5,RB Bragantino,8,2,6
6,Vasco da Gama,10,4,6
7,Corinthians,10,5,5
8,Juventude,8,3,5
9,Internacional,11,7,4
